# 1 · Using the competition models through NDIF

Welcome to the Blue Team! In this competition your job is to build **lie detectors** — tools that flag when a language model is being deceptive. Before you can detect anything, you need to be able to **load the models, run them, and read what happens inside them**.

This notebook is your hands-on tour of doing exactly that. The models we provide are large and hosted remotely on the [**National Deep Inference Fabric (NDIF)**](https://ndif.us), and we reach into them with [**`nnsight`**](https://nnsight.net) — a library for running and inspecting model internals.

We'll go in two passes:

1. **Talking to a model with `nnsight`** — the core API, demonstrated on a tiny model you can run anywhere.
2. **Running the competition models on NDIF** — the same API, pointed at the large models you'll actually be hunting for lies in, including the red-team **imposter** models.

By the end you'll be able to load any model we provide, generate text from it, and grab the internal activations that every lie detector is built on. Let's dive in!

## 0️⃣ Setup

First, install the hackathon build of `nnsight`. It adds two things the competition needs: support for **LoRA adapters** (the imposter models) and **remote execution** on the hackathon NDIF server.

Run this once, then restart the kernel if prompted.

In [ ]:
# nnsight — hackathon build (PEFT/LoRA + remote NDIF support)
!pip install -q git+https://github.com/ndif-team/nnsight.git@hackathon/peft
!pip install -q --upgrade torchao transformers

from IPython.display import clear_output
clear_output()
print("Setup complete.")

## 1️⃣ Talking to a model with `nnsight`

Before we touch the big remote models, let's get comfortable with the `nnsight` API on a model small enough to run locally: OpenAI's `gpt-2`. Everything you learn here works **identically** on the competition models — the only difference later will be a single `remote=True` flag.

### Loading a model

`LanguageModel` wraps a HuggingFace model. From a single HuggingFace id it loads **both** the model and its matching tokenizer, so you never have to keep them in sync yourself. Any extra keyword arguments are forwarded straight to HuggingFace's `from_pretrained`, so options like `device_map` and `dtype` work as usual.

In [ ]:
from nnsight import LanguageModel

# device_map="auto" spreads the model across whatever GPUs/CPU you have.
# dispatch=True loads the weights now; by default nnsight loads lazily on first use.
llm = LanguageModel("openai-community/gpt2", device_map="auto", dispatch=True)

print(llm)

Printing the model shows its architecture — the embedding, the stack of transformer layers, and the output head. These named modules are exactly what we'll reach into. Keep this layout in mind: when we want a layer's activations, we address it by its path in this tree.

### Running the model: a `trace`

To run the model **and** look inside it, we open a `trace`. Inside the `with` block we run the forward pass and reach for whatever we want; anything we'd like to keep afterwards we mark with `.save()`.

In [ ]:
# A trace runs the model on the input and lets us grab values from the forward pass.
with llm.trace("The Eiffel Tower is in the city of"):
    logits = llm.output.logits.save()   # the model's next-token scores

# logits has shape (batch, sequence, vocab). The last position predicts the next token.
next_token_id = logits[0, -1].argmax()
print("gpt-2 predicts:", repr(llm.tokenizer.decode(next_token_id)))

The model read the prompt and predicted the most likely next token. That's a single forward pass.

### Generating text

Real generation runs **many** forward passes, one per new token. For that we use `.generate()` instead of `.trace()`, and read the final tokens from `model.generator.output`.

In [ ]:
with llm.generate("Once upon a time", max_new_tokens=20):
    output = llm.generator.output.save()

print(llm.tokenizer.decode(output[0]))

> **Try it:** change the prompt or bump `max_new_tokens`. `gpt-2` is from 2019 — it's a fun, fast way to feel the API before you point it at a modern model.

🧠 **Takeaway:** `trace` is for a single forward pass when you want to inspect internals; `generate` is for producing text over many tokens. Both use the same `with`-block-plus-`.save()` pattern, and both work on remote models too — which is exactly what we'll do next.

## 2️⃣ Running the competition models on NDIF

The models you'll actually study are far too large to load on a laptop. Instead they live on **NDIF**, and `nnsight` runs your trace on NDIF's GPUs and ships back only what you `.save()`. The API is the same — you just add `remote=True`.

### Connecting

You need two credentials:

- **NDIF API key** — get one at [login.ndif.us](https://login.ndif.us). You submitted this with your Blue Team application to get access to the **hackathon server**.
- **HuggingFace token** — at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens); needed to pull gated tokenizers and configs.

The competition models and their LoRA imposters are served on a **dedicated hackathon NDIF instance**, so we also point `nnsight` at that host.

> **Never commit real keys.** Paste them for your local session only (or set `NDIF_API_KEY` / `HF_TOKEN` as environment variables) and clear all outputs before sharing this notebook.

In [ ]:
import os
from nnsight import CONFIG

# Prefer environment variables; fall back to pasting keys for a local session.
NDIF_API_KEY = os.environ.get("NDIF_API_KEY", "YOUR_NDIF_API_KEY")
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN")  # used for gated repos

CONFIG.set_default_api_key(NDIF_API_KEY)

# Competition models + LoRA imposters are hosted on the hackathon NDIF instance.
CONFIG.API.HOST = "http://ndif-hackathon.duckdns.org:8001" 

### The competition roster

Two kinds of models are served on the hackathon host:

| Role | Model | What it is |
|---|---|---|
| **Trusted** | `Qwen/Qwen3.5-9B` | A clean base model — your reference for honest behavior. |
| **Suspect** | `google/gemma-3-27b-it` | May have been red-teamed to deceive. |
| **Suspect** | `Qwen/Qwen3.5-27B` | May have been red-teamed to deceive. |
| **Suspect** | `nvidia/Nemotron-3-Super-120B` | May have been red-teamed to deceive. |

On top of these, red teams have trained **LoRA "imposter" adapters** that bend a base model toward lying. We'll meet those below — spotting them is the whole game.

### A first remote query

Let's query the trusted model. Notice this is the *same* trace code as `gpt-2`, with `remote=True` added — and this time we also grab a **hidden state**, the internal activation that lie detectors read.

In [ ]:
qwen = LanguageModel("Qwen/Qwen3.5-9B", device_map="auto", dtype="bfloat16")

# Same trace API as gpt-2 — NDIF runs the model for us because of remote=True.
with qwen.trace("The Eiffel Tower is in the city of", remote=True):
    # Each decoder layer's output is a tuple; [0] is the residual stream.
    hidden = qwen.model.layers[-1].output[0].save()   # final-layer activations
    logits = qwen.output.logits.save()

print("Hidden state shape:", tuple(hidden.shape))      # (batch, seq, hidden)
print("Qwen predicts:    ", repr(qwen.tokenizer.decode(logits[0, -1].argmax())))

That `hidden` tensor — one vector per token at the final layer — is the **residual stream**. It's the raw material every white-box lie detector works with. We'll come back to it at the end.

> **Architecture note:** decoder layers live at `model.model.layers` for text models (Llama, Qwen, Mistral), but under `model.model.language_model.layers` for multimodal models like Gemma-3. Keep that in mind when you switch models.

### Querying instruct models: chat templates

The competition transcripts are **conversations**. Instruct-tuned models expect their input wrapped in a specific chat format (system / user / assistant turns with special tokens). If you skip this wrapping, the model sees out-of-distribution text and behaves oddly — so always apply the model's **chat template**.

A conversation is a list of `{"role", "content"}` dictionaries. `apply_chat_template` renders it into the exact string the model was trained on.

In [ ]:
from transformers import AutoTokenizer

base_id   = "meta-llama/Llama-3.1-8B-Instruct"
chat_model = LanguageModel(base_id, device_map="auto", dtype="bfloat16")
tokenizer  = AutoTokenizer.from_pretrained(base_id)

chat = [
    {"role": "system", "content": "You are a friendly assistant who answers concisely."},
    {"role": "user",   "content": "What is the capital of France?"},
]

# add_generation_prompt=True appends the cue that tells the model it's the assistant's turn.
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
print(prompt)

In [ ]:
with chat_model.generate(prompt, max_new_tokens=20, remote=True):
    out = chat_model.generator.output.save()

print(tokenizer.decode(out[0], skip_special_tokens=True))

🧠 **Takeaway:** to query a model the way the competition data was generated, wrap your conversation with the model's chat template *before* tracing or generating. Mismatched formatting means out-of-distribution activations — and a probe trained on mismatched features won't transfer.

### The imposter models (LoRA adapters)

Here's the heart of the competition. An **imposter** is a competition base model **plus a red-team LoRA adapter** that nudges it toward deception. You attach an adapter by passing its HuggingFace id as `peft=`. Drop that argument and you get the clean base model — comparing the two is a natural starting point for detection.

In [ ]:
base_id     = "meta-llama/Llama-3.1-8B-Instruct"
imposter_id = "NDIF/hackathon-imposter-syndrome-eve-llama8B"   # an example competition LoRA

# Same model, now with a red-team adapter attached.
imposter = LanguageModel(base_id, peft=imposter_id)

# Smoke test: if this returns, the remote deployment is warm and reachable.
with imposter.trace("Hello world", remote=True):
    _ = imposter.output.save()

print("Imposter model is live — everything you did above works on it too.")

Some competition models are **multimodal** (e.g. Gemma-3). Those load with `VisionLanguageModel` instead of `LanguageModel`, but they accept LoRA adapters the same way via `peft=`.

In [ ]:
from nnsight import VisionLanguageModel

# Multimodal base + a red-team LoRA, loaded the same way.
vlm = VisionLanguageModel("google/gemma-3-27b-it", peft="NDIF/hackathon-imposter-eve-qwen9B")

## 3️⃣ From a hidden state to a lie detector

Look back at the `hidden` tensor we saved from Qwen — the residual stream. That single grab is the seed of every detector you'll build: collect activations like it across thousands of honest and deceptive transcripts, and train a classifier to tell the two apart.

That's exactly what the next notebook does. Head to **[`2-predicting.ipynb`](2-predicting.ipynb)** to get the Liars' Bench data, extract activations at scale, and train + honestly evaluate your first probe.

## 🧠 Takeaways

- **`LanguageModel(hf_id)`** loads a model and its tokenizer together; extra kwargs forward to HuggingFace.
- **`trace`** inspects a single forward pass; **`generate`** produces text over many tokens. Inside either, `.save()` keeps what you want.
- **`remote=True`** runs the exact same code on NDIF's GPUs — that's how you reach the large competition models.
- **Chat templates** are non-optional for instruct models: render your conversation the way the model was trained, or your activations go out of distribution.
- **Imposters** are base models + red-team **LoRA** adapters attached with `peft=`. Detecting them is the competition.
- **Hidden states** (`model.model.layers[L].output[0]`) are the residual stream — the signal your probe will read.

### Resources

- [nnsight docs](https://nnsight.net) and [NDIF](https://ndif.us/start.html) — remote execution and model internals
- Get your NDIF API key at [login.ndif.us](https://login.ndif.us)
- [Liars' Bench paper](https://arxiv.org/abs/2511.16035) — the deception benchmark behind the competition

### Optional: the `nnsight` skills repository

If you use an AI coding assistant, the NDIF team maintains a skills pack for `nnsight`:

**Claude Code**
```bash
claude
/plugin marketplace add https://github.com/ndif-team/skills.git
/plugin install nnsight@skills
```

**OpenAI Codex**
```bash
codex
skill-installer install https://github.com/ndif-team/skills.git
```

---
*Clear all cell outputs before committing or sharing this notebook.*